# Baselines: random vs gene-level split

Purpose: quantify how much of the model's apparent skill under random split is due to data leakage vs. genuine learning.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src import baselines
from src.splits import random_split, gene_level_split
from src.utils import set_seed

set_seed(42)
df = pd.read_csv('../data/expression_utr_summary.csv')
print(f'{len(df)} rows, {df.gene_symbol.nunique()} genes')

In [ ]:
# ── Random split ───────────────────────────────────────────────
subs = random_split(df, val_fraction=0.1, seed=42)
results_random = baselines.run_all(subs['train'], subs['val'])

# ── Gene-level split ───────────────────────────────────────────
subs = gene_level_split(df, val_fraction=0.1, seed=42)
results_gene = baselines.run_all(subs['train'], subs['val'])

In [ ]:
import pandas as pd

rows = []
for name in ['global_mean', 'tissue_mean', 'gc_length_tissue_ridge', 'kmer4_tissue_ridge']:
    rows.append({
        'Baseline': name,
        'R² (random)':  round(results_random[name]['R2'], 4),
        'R² (gene)':    round(results_gene[name]['R2'], 4),
        'MAPE % (random)': round(results_random[name]['MAPE_percent'], 2),
        'MAPE % (gene)':   round(results_gene[name]['MAPE_percent'], 2),
    })
pd.DataFrame(rows).set_index('Baseline')

## Reading the numbers

- `4-mer + tissue (Ridge)` gets ≈+0.07 R² on random split but only ≈+0.025 on gene-level. That gap is the data leakage.
- All gene-level R² values sit close to the `tissue_mean` floor of ≈0.015. This tells us that on native ProteomicsDB data with 1738 unique genes, hand-crafted sequence features carry almost no predictive signal beyond the tissue context.
- Whether UTR-BERT can extract more than this — and by how much — is the empirical question the neural model exists to answer. But the ceiling is bounded by the between-gene variance (70% of total; see EDA notebook).